In [43]:
import tensorflow as tf
from tensorflow.keras.models import load_model
import pickle
import pandas as pd
import numpy as np

In [44]:
### Load the trained model, scaler pickle, onehot

model = load_model('model.h5')

## load the encoder and scaler
with open('onehot_encoder_geo.pk1', 'rb') as file:
  onehot_encoder_geo = pickle.load(file)

with open('label_encoder_gender.pk1', 'rb') as file:
  label_encoder_gender = pickle.load(file)

with open('scaler.pk1', 'rb') as file:
  scaler = pickle.load(file)

In [45]:
## Example input data
input_data = {
  'CreditScore': 600,
  'Geography': 'France',
  'Gender': 'Male',
  'Age': 40,
  'Tenure': 3,
  'Balance': 60000,
  'NumOfProducts': 2,
  'HasCrCard': 1,
  'IsActiveMember': 1,
  'EstimatedSalary': 50000
}

In [46]:
import pandas as pd

input_df = pd.DataFrame([input_data])

In [47]:
print(onehot_encoder_geo.categories_)

[array(['France', 'Germany', 'Spain'], dtype=object)]


In [48]:
# One hot encode 'Geography'

geo_encoded = onehot_encoder_geo.transform(input_df[['Geography']])
geo_encoded_df = pd.DataFrame(geo_encoded, columns=onehot_encoder_geo.get_feature_names_out(['Geography']))
geo_encoded_df


,Geography_France,Geography_Germany,Geography_Spain
0,1.0,0.0,0.0


In [51]:
# Build a fresh DataFrame each time so this cell can be rerun safely
prediction_df = pd.DataFrame([input_data])

# Encode Geography
geo_encoded = onehot_encoder_geo.transform(prediction_df[['Geography']])
geo_encoded_df = pd.DataFrame(
    geo_encoded,
    columns=onehot_encoder_geo.get_feature_names_out(['Geography'])
)

# Encode Gender
prediction_df['Gender'] = label_encoder_gender.transform(prediction_df['Gender'])

# Combine all features in the same order used when fitting the scaler
processed_df = pd.concat([prediction_df.drop('Geography', axis=1), geo_encoded_df], axis=1)
processed_df = processed_df[scaler.feature_names_in_]

# Scale and predict
input_scaled = scaler.transform(processed_df)
prediction = model.predict(input_scaled)
prediction_probability = prediction[0][0]

print(f'Churn probability: {prediction_probability:.4f}')
print('The customer is likely to churn.' if prediction_probability > 0.5 else 'The customer is not likely to churn.')

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 67ms/step

1/1 ━━━━━━━━━━━━━━━━━━━━ 0s 136ms/step
Churn probability: 0.0477
The customer is not likely to churn.
